# Two Pointers — Interactive Teach-Back
**Harbour Phase 0 · Python Problem Solving & Complexity · Chunk 2**

Main topics:
1. Sorted arrays — target pair
2. Palindromes
3. Container With Most Water

Teaching flow: **problem → straightforward solution → complexity → Two Pointers reasoning → optimized solution → comparison**.

> **Core idea:** Two Pointers keeps two meaningful positions and moves one or both according to a rule that lets us safely eliminate unnecessary work.

The key question is: **What fact makes it safe to move this pointer and never reconsider what was discarded?**


## 1. Sorted Array — Pair With a Target Sum

Given a **sorted** array and a target, find two different elements whose sum equals the target.

### Straightforward approach: try every pair
This is correct, but may inspect roughly `n(n-1)/2` pairs.

- Time: `O(n²)`
- Extra space: `O(1)`


In [ ]:
numbers = [1, 2, 3, 4, 6, 8, 9]
target = 12

def pair_sum_brute_force(numbers, target):
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] + numbers[j] == target:
                return (numbers[i], numbers[j])
    return None

result = pair_sum_brute_force(numbers, target)
print(result)

### Two Pointers

Start at both ends.

```text
L                       R
↓                       ↓
[1, 2, 3, 4, 6, 8, 9]
```

- Sum equals target → found.
- Sum too small → move `left` right.
- Sum too large → move `right` left.

**Why safe?** If the current left value plus the largest available value is still too small, that left value cannot work with any smaller partner. The reverse reasoning applies when the sum is too large.

- Time: `O(n)`
- Extra space: `O(1)`


In [ ]:

def pair_sum_two_pointers(numbers, target):
    left = 0
    right = len(numbers) - 1

    while left < right:
        current_sum = numbers[left] + numbers[right]

        if current_sum == target:
            return (numbers[left], numbers[right])

        if current_sum < target:
            left += 1
        else:
            right -= 1

    return None

result = pair_sum_two_pointers(numbers, target)
print(result)

In [ ]:
# Live experiment: change the target and rerun.
target = 15

brute_result, brute_checks = pair_sum_brute_force(numbers, target)
pointer_result, pointer_checks = pair_sum_two_pointers(numbers, target, trace=True)

print("\nBrute force :", brute_result, "| checks:", brute_checks)
print("Two Pointers:", pointer_result, "| checks:", pointer_checks)


### Sorted-array comparison

| Approach | Time | Extra space |
|---|---:|---:|
| Every pair | `O(n²)` | `O(1)` |
| Two Pointers | **`O(n)`** | `O(1)` |

The movement rule depends on **sorted order**. If input is unsorted, sorting first costs `O(n log n)`. Hashing may instead offer average `O(n)` time with `O(n)` extra memory.


## 2. Palindrome

A palindrome reads the same forward and backward.

### Straightforward Python approach: reverse the string
This works, but creates another string.

- Time: `O(n)`
- Extra space: `O(n)`


In [ ]:
def palindrome_reverse(text):
    reversed_text = text[::-1]
    return text == reversed_text

result = palindrome_reverse("racecar")
print(result)


### Two Pointers

A palindrome requires mirrored positions to match.

```text
R A C E C A R
↑           ↑
L           R
```

Compare both ends and move inward. One mismatch proves the answer is `False`.

**Invariant:** everything already passed by the pointers consists of matching mirrored pairs.

- Time: `O(n)`
- Extra space: `O(1)`

So here Two Pointers mainly improves **space**, not Big-O time.


In [ ]:
def palindrome_two_pointers(text):
    left = 0
    right = len(text) - 1

    while left < right:
        if text[left] != text[right]:
            return False

        left += 1
        right -= 1

    return True

result = palindrome_two_pointers("racecar")
print(result)


### Variation: ignore punctuation and case

Instead of constructing a cleaned copy, the pointers can skip unwanted characters as they move.


In [ ]:
def valid_palindrome_two_pointers(text):
    left = 0
    right = len(text) - 1

    while left < right:
        while left < right and not text[left].isalnum():
            left += 1

        while left < right and not text[right].isalnum():
            right -= 1

        if text[left].lower() != text[right].lower():
            return False

        left += 1
        right -= 1

    return True

result = valid_palindrome_two_pointers("A man, a plan, a canal: Panama")
print(result)

### Palindrome comparison

| Approach | Time | Extra space |
|---|---:|---:|
| Reverse string | `O(n)` | `O(n)` |
| Two Pointers | `O(n)` | **`O(1)`** |

Teaching question: **Why `left < right` rather than `left <= right`?**  
When both pointers meet at the middle character, that character trivially equals itself.


## 3. Container With Most Water

Given wall heights, choose two walls that maximize:

```text
area = width × usable height
width = right - left
usable height = min(left height, right height)
```

The shorter wall limits the water level.

### Straightforward approach: check every pair

- Time: `O(n²)`
- Extra space: `O(1)`


In [ ]:
heights = [1, 8, 6, 2, 5, 4, 8, 3, 7]

def max_water_brute_force(heights):
    max_area = 0

    for left in range(len(heights)):
        for right in range(left + 1, len(heights)):
            width = right - left
            usable_height = min(heights[left], heights[right])
            area = width * usable_height
            max_area = max(max_area, area)

    return max_area

result = max_water_brute_force(heights)
print(result)

### Two Pointers

Start with maximum possible width.

```text
L                               R
↓                               ↓
[1, 8, 6, 2, 5, 4, 8, 3, 7]
```

After calculating area, move the pointer at the **shorter wall**.

#### Why?

If the walls are `5` and `10`, usable height is only `5`. Keeping the `5` while moving the `10` inward makes width smaller while usable height remains at most `5`.

Discarding the shorter wall is the only move that gives the limiting height a chance to increase.

> Moving the shorter wall does **not guarantee** the next area is larger. It preserves the only possibility of overcoming the current height limit.


In [ ]:

heights = [1, 8, 6, 2, 5, 4, 8, 3, 7]

def max_water_two_pointers(heights):
    left = 0
    right = len(heights) - 1
    max_area = 0

    while left < right:
        width = right - left
        usable_height = min(heights[left], heights[right])
        area = width * usable_height

        max_area = max(max_area, area)

        if heights[left] < heights[right]:
            left += 1
        else:
            right -= 1

    return max_area

result = max_water_two_pointers(heights)
print(result)

In [ ]:
# Direct comparison
brute_area, brute_checks = max_water_brute_force(heights)
pointer_area, pointer_checks = max_water_two_pointers(heights)

print("Brute force  → area:", brute_area, "| checks:", brute_checks)
print("Two Pointers → area:", pointer_area, "| checks:", pointer_checks)

assert brute_area == pointer_area


### Container comparison

| Approach | Time | Extra space |
|---|---:|---:|
| Every pair | `O(n²)` | `O(1)` |
| Two Pointers | **`O(n)`** | `O(1)` |

**Trap question:** Why not move the taller wall?  
Because the shorter wall is already limiting usable height. Moving the taller wall only reduces width without allowing that limit to increase.


# 4. Main Comparison

| Problem | Straightforward approach | Two Pointers | Main improvement |
|---|---|---|---|
| Sorted pair sum | `O(n²)` time / `O(1)` space | `O(n)` / `O(1)` | Time |
| Palindrome | `O(n)` time / `O(n)` space | `O(n)` / `O(1)` | Space |
| Container | `O(n²)` time / `O(1)` space | `O(n)` / `O(1)` | Time |

The movement rule changes by problem:

- **Sorted pair sum:** too small → left; too large → right.
- **Palindrome:** match → both inward; mismatch → stop.
- **Container:** move the shorter wall.

The common principle is **safe elimination**, not a memorized movement.


# 5. Optional Exploration — Same-Direction Pointers

A scanner can move through the input while another pointer maintains a result boundary.

Explore:
- Remove duplicates from a sorted array
- Move zeroes
- Partition an array

```text
write
 ↓
[1, 1, 2, 2, 3, 4, 4]
    ↑
   read

Both move →
```


In [ ]:
def remove_duplicates(numbers):
    if not numbers:
        return 0

    write = 1
    for read in range(1, len(numbers)):
        if numbers[read] != numbers[read - 1]:
            numbers[write] = numbers[read]
            write += 1

    return write

values = [1, 1, 2, 2, 3, 4, 4]
k = remove_duplicates(values)
print("Unique portion:", values[:k])


# 6. Optional Exploration — Fast/Slow Pointers

```text
slow → one step
fast → two steps
```

Explore:
- Middle of a linked list
- Linked-list cycle detection
- Start of a cycle

Useful terms: **tortoise and hare**, **Floyd's cycle detection**, **fast/slow pointers**.

These are included only to show that Two Pointers is broader than opposite-end `left/right` problems.


# 7. Teach-Back Checklist

Before presenting, make sure you can explain:

- What Two Pointers means without mentioning code.
- Why sorted order makes target-sum elimination safe.
- Why palindrome improves space rather than Big-O time.
- Why Container With Most Water moves the shorter wall.
- Why Two Pointers does not always mean opposite ends.
- Why Two Pointers does not automatically mean `O(n)`.
- When hashing might be preferable.
- The invariant/elimination reasoning behind each main example.

## Final takeaway

> **Two Pointers is not powerful because it uses two variables. It is powerful when the structure of the problem gives us enough information to move through the search space without reconsidering discarded possibilities.**


# 8. Quick Live Q&A

1. In sorted pair sum, why does a sum that is too small justify moving `left`?
2. Why can't that same movement rule be trusted on an unsorted array?
3. Why is reversing a string `O(n)` extra space?
4. What does Two Pointers improve in the palindrome example?
5. What two factors determine container area?
6. Why is the shorter wall the useful one to discard?
7. Is every Two Pointers algorithm `O(n)`?
8. How do same-direction pointers differ from opposite-direction pointers?
